# Rwanda House Price Prediction 
**names:** Nelly Stella Nzeyimana

This notebook develops a multiple linear regression model for predicting `House_Price_Million_RWF` and follows the assignment workflow from raw-data exploration through deployment preparation.

## 1. Load the Raw Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

data = pd.read_csv("house_price_prediction_dataset_raw.csv")
data.head()

In [ ]:
print("Shape:", data.shape)
print("\nMissing values:")
display(data.isna().sum().to_frame("Missing"))
print("\nDuplicate rows:", data.duplicated().sum())
display(data.describe(include="all").T)

## 2. Cleaning

In [ ]:
clean = data.drop_duplicates().copy()
numeric_fields = ["Area_m2","Bedrooms","Bathrooms","House_Age_Years","Distance_to_City_km","Parking_Spaces","House_Price_Million_RWF"]
for col in numeric_fields:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")
for col in numeric_fields[:-1]:
    clean[col] = clean[col].fillna(clean[col].median())
clean["Neighborhood"] = clean["Neighborhood"].fillna(clean["Neighborhood"].mode()[0])
clean = clean.dropna(subset=["House_Price_Million_RWF"])
for col in numeric_fields:
    clean = clean[clean[col] >= 0]

print("Cleaned shape:", clean.shape)
print("Remaining missing cells:", int(clean.isna().sum().sum()))
print("Remaining duplicates:", int(clean.duplicated().sum()))

### IQR outlier screening

In [ ]:
outlier_rows=[]
for col in numeric_fields:
    q1=clean[col].quantile(.25); q3=clean[col].quantile(.75); iqr=q3-q1
    low=q1-1.5*iqr; high=q3+1.5*iqr
    outlier_rows.append([col,low,high,int(((clean[col]<low)|(clean[col]>high)).sum())])
display(pd.DataFrame(outlier_rows,columns=["Variable","Lower","Upper","IQR Outliers"]))

## 3. Exploratory Analysis

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(clean["House_Price_Million_RWF"], bins=12, edgecolor="black")
plt.xlabel("House Price (Million RWF)"); plt.ylabel("Frequency"); plt.title("Distribution of House Prices")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(clean["Area_m2"],clean["House_Price_Million_RWF"],alpha=.8)
plt.xlabel("Area (m²)"); plt.ylabel("House Price (Million RWF)"); plt.title("Area vs House Price")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
clean.boxplot(column="House_Price_Million_RWF",by="Neighborhood",grid=False)
plt.title("House Price by Neighborhood"); plt.suptitle(""); plt.xlabel("Neighborhood"); plt.ylabel("House Price (Million RWF)"); plt.xticks(rotation=25)
plt.show()

In [ ]:
corr_cols=["Area_m2","Bedrooms","Bathrooms","House_Age_Years","Distance_to_City_km","Parking_Spaces","House_Price_Million_RWF"]
display(clean[corr_cols].corr().round(3))

## 4. Prepare Data

In [ ]:
target="House_Price_Million_RWF"
predictors=["Area_m2","Bedrooms","Bathrooms","House_Age_Years","Distance_to_City_km","Parking_Spaces","Neighborhood"]
X=clean[predictors]; y=clean[target]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42)
numeric_features=predictors[:-1]; categorical_features=["Neighborhood"]
print("Training:",len(X_train),"Testing:",len(X_test))

## 5. Train Multiple Linear Regression

In [ ]:
preprocessor=ColumnTransformer([
("numeric",Pipeline([("imputer",SimpleImputer(strategy="median"))]),numeric_features),
("categorical",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(drop="first",handle_unknown="ignore"))]),categorical_features)
])
model=Pipeline([("preprocessor",preprocessor),("regressor",LinearRegression())])
model.fit(X_train,y_train)
print("Model trained successfully.")

## 6. Coefficients and Multicollinearity

In [ ]:
names=model.named_steps["preprocessor"].get_feature_names_out()
display(pd.DataFrame({"Variable":names,"Coefficient":model.named_steps["regressor"].coef_}))
print("Intercept:",model.named_steps["regressor"].intercept_)
display(clean[numeric_features].corr().round(3))

## 7. Evaluation

In [ ]:
train_pred=model.predict(X_train); test_pred=model.predict(X_test)
p=len(model.named_steps["preprocessor"].get_feature_names_out())
def adjusted_r2(ytrue,pred):
    r2=r2_score(ytrue,pred); n=len(ytrue)
    return 1-(1-r2)*(n-1)/(n-p-1)
evaluation=pd.DataFrame({
"Dataset":["Training","Testing"],
"R²":[r2_score(y_train,train_pred),r2_score(y_test,test_pred)],
"Adjusted R²":[adjusted_r2(y_train,train_pred),adjusted_r2(y_test,test_pred)],
"MAE (Million RWF)":[mean_absolute_error(y_train,train_pred),mean_absolute_error(y_test,test_pred)],
"RMSE (Million RWF)":[mean_squared_error(y_train,train_pred)**.5,mean_squared_error(y_test,test_pred)**.5]})
display(evaluation.round(4))

## 8. Diagnostics

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(y_test,test_pred,alpha=.8)
lo=min(y_test.min(),test_pred.min()); hi=max(y_test.max(),test_pred.max())
plt.plot([lo,hi],[lo,hi],"--")
plt.xlabel("Actual Price (Million RWF)"); plt.ylabel("Predicted Price (Million RWF)"); plt.title("Actual vs Predicted")
plt.show()

In [ ]:
residuals=y_test.to_numpy()-test_pred
plt.figure(figsize=(8,5))
plt.scatter(test_pred,residuals,alpha=.8); plt.axhline(0,linestyle="--")
plt.xlabel("Predicted Price (Million RWF)"); plt.ylabel("Residual"); plt.title("Residuals vs Predicted")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(residuals,bins=10,edgecolor="black")
plt.xlabel("Residual (Million RWF)"); plt.ylabel("Frequency"); plt.title("Residual Distribution")
plt.show()

## 9. Save, Reload and Predict

In [ ]:
joblib.dump(model,"house_price_model_nelly_stella.sav")
loaded_model=joblib.load("house_price_model_nelly_stella.sav")
print("Reload check:",np.allclose(model.predict(X_test),loaded_model.predict(X_test)))

In [ ]:
example_neighborhood=sorted(clean["Neighborhood"].astype(str).unique())[0]
new_house=pd.DataFrame({
"Area_m2":[180],"Bedrooms":[4],"Bathrooms":[3],"House_Age_Years":[5],
"Distance_to_City_km":[4],"Parking_Spaces":[2],"Neighborhood":[example_neighborhood]
})
prediction=loaded_model.predict(new_house)[0]
print(f"Predicted house price: {prediction:.2f} million RWF")

## 10. Conclusion
The model provides an estimation of house price from the available predictors. Test-set performance and residual diagnostics should be considered when interpreting its generalisation.